# Combined XGBoost Inference Workflow for `y_fail_target` and `y_hf_fail_target`

This notebook combines both saved XGBoost models into a single shared workflow.

It does four things:

1. Rebuilds the exact pair-feature dataset used in training  
2. Trains and saves **two separate full pipelines**
   - `y_fail_target`
   - `y_hf_fail_target`
3. Stores each model in its **own directory**
4. Provides shared inference helpers for:
   - single-pair scoring
   - multi-target ranking
   - scoring both models side by side


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:

import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import pathlib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report


## 1. Paths

Update these paths for your environment. The two models are stored in separate directories.


In [ ]:

# --- EDIT THESE PATHS ---
BASE_DATA_PATH = r"/content/drive/MyDrive/Practicum/incident_outputs/barrier_model_dataset_base_v3.csv"
MODELS_ROOT    = r"/content/drive/MyDrive/Practicum/Models"

Y_FAIL_DIR     = os.path.join(MODELS_ROOT, "y_fail_target")
Y_HF_FAIL_DIR  = os.path.join(MODELS_ROOT, "y_hf_fail_target")

os.makedirs(Y_FAIL_DIR, exist_ok=True)
os.makedirs(Y_HF_FAIL_DIR, exist_ok=True)

MODEL_REGISTRY = {
    "y_fail_target": {
        "model_dir": Y_FAIL_DIR,
        "model_path": os.path.join(Y_FAIL_DIR, "xgb_y_fail_target_pipeline.joblib"),
        "meta_path": os.path.join(Y_FAIL_DIR, "xgb_y_fail_target_metadata.json"),
    },
    "y_hf_fail_target": {
        "model_dir": Y_HF_FAIL_DIR,
        "model_path": os.path.join(Y_HF_FAIL_DIR, "xgb_y_hf_fail_target_pipeline.joblib"),
        "meta_path": os.path.join(Y_HF_FAIL_DIR, "xgb_y_hf_fail_target_metadata.json"),
    },
}

MODEL_REGISTRY


{'y_fail_target': {'model_dir': '/content/drive/MyDrive/Practicum/Models/y_fail_target',
  'model_path': '/content/drive/MyDrive/Practicum/Models/y_fail_target/xgb_y_fail_target_pipeline.joblib',
  'meta_path': '/content/drive/MyDrive/Practicum/Models/y_fail_target/xgb_y_fail_target_metadata.json'},
 'y_hf_fail_target': {'model_dir': '/content/drive/MyDrive/Practicum/Models/y_hf_fail_target',
  'model_path': '/content/drive/MyDrive/Practicum/Models/y_hf_fail_target/xgb_y_hf_fail_target_pipeline.joblib',
  'meta_path': '/content/drive/MyDrive/Practicum/Models/y_hf_fail_target/xgb_y_hf_fail_target_metadata.json'}}

## 2. Recreate the exact training feature setup

This section mirrors the logic from the original training notebook so the inference feature contract stays identical.


In [ ]:

# ======================================================
# Feature definitions copied from the original notebook
# ======================================================

BARRIER_FEATURES_TARGET = [
    "barrier_level",
    "lod_industry_standard",
    "pathway_sequence",
    "lod_numeric",
    "num_threats_in_lod_numeric"
]

BARRIER_FEATURES_COND = [
    "barrier_level",
    "lod_industry_standard",
    "barrier_condition",
    "pathway_sequence",
    "lod_numeric",
    "num_threats_in_lod_numeric"
]

CONTEXT_FEATURES = [
    "total_prev_barriers_incident",
    "total_mit_barriers_incident",
    "num_threats_in_sequence",
    "flag_environmental_threat",
    "flag_electrical_failure",
    "flag_procedural_error",
    "flag_mechanical_failure",
]

ID_COLS = ["incident_id", "control_id"]
LABEL_COLS = ["y_fail", "y_hf_fail"]

ALL_RAW_FEATURES = sorted(list(set(
    BARRIER_FEATURES_TARGET +
    BARRIER_FEATURES_COND
)))

KEEP_COLS = ID_COLS + ALL_RAW_FEATURES + CONTEXT_FEATURES + LABEL_COLS


In [ ]:

def build_pair_dataset(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str], list[str]]:
    """Rebuild the pair dataset exactly like the training notebook."""

    df_clean = (
        df[KEEP_COLS]
        .dropna(subset=ALL_RAW_FEATURES + CONTEXT_FEATURES + LABEL_COLS)
        .copy()
        .reset_index(drop=True)
    )

    tgt = df_clean.copy()
    cond = df_clean.copy()

    tgt = tgt.rename(columns={c: f"{c}_target" for c in BARRIER_FEATURES_TARGET + LABEL_COLS})

    cond = cond.rename(columns={c: f"{c}_cond" for c in BARRIER_FEATURES_COND})
    cond = cond.rename(columns={"y_fail": "y_fail_cond", "y_hf_fail": "y_hf_fail_cond"})
    cond = cond.rename(columns={"control_id": "control_id_cond"})

    df_pairs = tgt.merge(
        cond.drop(columns=CONTEXT_FEATURES),
        on="incident_id",
        how="inner"
    )

    df_pairs = df_pairs[df_pairs["control_id"] != df_pairs["control_id_cond"]].copy()
    df_pairs = df_pairs[df_pairs["y_fail_cond"] == 1].reset_index(drop=True)

    cat_target = [
        "lod_industry_standard_target",
        "barrier_level_target",
    ]

    num_target = [
        "pathway_sequence_target",
        "lod_numeric_target",
        "num_threats_in_lod_numeric_target",
    ]

    cat_cond = [
        "lod_industry_standard_cond",
        "barrier_level_cond",
        "barrier_condition_cond",
    ]

    num_cond = [
        "pathway_sequence_cond",
        "lod_numeric_cond",
        "num_threats_in_lod_numeric_cond",
    ]

    cat_context = []
    num_context = CONTEXT_FEATURES.copy()

    cat_all = cat_target + cat_cond + cat_context
    num_all = num_target + num_cond + num_context

    all_features = cat_all + num_all
    return df_pairs, cat_all, num_all, all_features


## 3. Shared model-building utilities

In [ ]:

def assign_risk_tier(prob: float) -> str:
    if prob >= 0.66:
        return "HIGH"
    elif prob >= 0.33:
        return "MEDIUM"
    return "LOW"


def make_xgb_pipeline(cat_all: list[str], num_all: list[str], scale_pos_weight: float = 1.0) -> Pipeline:
    prep = ColumnTransformer([
        (
            "cat_ord",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            cat_all
        ),
        (
            "num_pass",
            "passthrough",
            num_all
        ),
    ])

    clf = xgb.XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

    return Pipeline([
        ("prep", prep),
        ("clf", clf)
    ])


def train_save_one_target(
    df_pairs: pd.DataFrame,
    cat_all: list[str],
    num_all: list[str],
    all_features: list[str],
    target_col: str,
    model_path: str,
    meta_path: str,
) -> dict:
    """Train, save, and return metadata for one target."""

    X = df_pairs[all_features].copy()
    y = df_pairs[target_col].copy()

    scale_pos_weight = (1 - y.mean()) / y.mean()

    pipeline = make_xgb_pipeline(cat_all, num_all, scale_pos_weight=scale_pos_weight)
    pipeline.fit(X, y)

    joblib.dump(pipeline, model_path)

    metadata = {
        "model_name": pathlib.Path(model_path).stem,
        "target": target_col,
        "model_type": "XGBClassifier",
        "categorical_features": cat_all,
        "numerical_features": num_all,
        "all_features": all_features,
        "risk_tier_thresholds": {
            "HIGH": ">= 0.66",
            "MEDIUM": ">= 0.33 and < 0.66",
            "LOW": "< 0.33"
        },
        "training_rows": int(len(X)),
        "positive_rate": float(y.mean()),
        "scale_pos_weight": float(scale_pos_weight)
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return metadata


def holdout_check(
    df_pairs: pd.DataFrame,
    cat_all: list[str],
    num_all: list[str],
    all_features: list[str],
    target_col: str,
) -> tuple[float, str]:
    """Optional quick validation check for one target."""

    X = df_pairs[all_features].copy()
    y = df_pairs[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    spw = (1 - y_train.mean()) / y_train.mean()
    pipe = make_xgb_pipeline(cat_all, num_all, scale_pos_weight=spw)
    pipe.fit(X_train, y_train)

    proba = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)

    tiers = pd.Series(proba).map(assign_risk_tier)
    y_pred = tiers.isin(["HIGH", "MEDIUM"]).astype(int)

    report = classification_report(
        y_test,
        y_pred,
        target_names=["No Event", "Event"]
    )

    return auc, report


## 4. Build the pair dataset once, then train both models

In [ ]:

# Load base dataset
df_base = pd.read_csv(BASE_DATA_PATH)

# Rebuild pair dataset once
df_pairs, CAT_ALL, NUM_ALL, ALL_FEATURES = build_pair_dataset(df_base)

print("Pair dataset shape:", df_pairs.shape)
print("Categorical features:", CAT_ALL)
print("Numerical features:", NUM_ALL)
print("Total model features:", len(ALL_FEATURES))


Pair dataset shape: (901, 26)
Categorical features: ['lod_industry_standard_target', 'barrier_level_target', 'lod_industry_standard_cond', 'barrier_level_cond', 'barrier_condition_cond']
Numerical features: ['pathway_sequence_target', 'lod_numeric_target', 'num_threats_in_lod_numeric_target', 'pathway_sequence_cond', 'lod_numeric_cond', 'num_threats_in_lod_numeric_cond', 'total_prev_barriers_incident', 'total_mit_barriers_incident', 'num_threats_in_sequence', 'flag_environmental_threat', 'flag_electrical_failure', 'flag_procedural_error', 'flag_mechanical_failure']
Total model features: 18


In [ ]:

# Optional hold-out checks
for target_col in ["y_fail_target", "y_hf_fail_target"]:
    auc, report = holdout_check(df_pairs, CAT_ALL, NUM_ALL, ALL_FEATURES, target_col)
    print("=" * 80)
    print(f"Target: {target_col}")
    print(f"Hold-out ROC-AUC: {auc:.4f}")
    print(report)


Target: y_fail_target
Hold-out ROC-AUC: 0.8709
              precision    recall  f1-score   support

    No Event       0.81      0.62      0.70        78
       Event       0.75      0.89      0.82       103

    accuracy                           0.77       181
   macro avg       0.78      0.75      0.76       181
weighted avg       0.78      0.77      0.77       181

Target: y_hf_fail_target
Hold-out ROC-AUC: 0.8300
              precision    recall  f1-score   support

    No Event       0.92      0.72      0.81       144
       Event       0.41      0.76      0.53        37

    accuracy                           0.73       181
   macro avg       0.67      0.74      0.67       181
weighted avg       0.82      0.73      0.75       181



In [ ]:

# Train and save both full pipelines
saved_metadata = {}

for target_col in ["y_fail_target", "y_hf_fail_target"]:
    cfg = MODEL_REGISTRY[target_col]
    metadata = train_save_one_target(
        df_pairs=df_pairs,
        cat_all=CAT_ALL,
        num_all=NUM_ALL,
        all_features=ALL_FEATURES,
        target_col=target_col,
        model_path=cfg["model_path"],
        meta_path=cfg["meta_path"],
    )
    saved_metadata[target_col] = metadata
    print(f"Saved {target_col} pipeline to: {cfg['model_path']}")
    print(f"Saved {target_col} metadata to: {cfg['meta_path']}")


Saved y_fail_target pipeline to: /content/drive/MyDrive/Practicum/Models/y_fail_target/xgb_y_fail_target_pipeline.joblib
Saved y_fail_target metadata to: /content/drive/MyDrive/Practicum/Models/y_fail_target/xgb_y_fail_target_metadata.json
Saved y_hf_fail_target pipeline to: /content/drive/MyDrive/Practicum/Models/y_hf_fail_target/xgb_y_hf_fail_target_pipeline.joblib
Saved y_hf_fail_target metadata to: /content/drive/MyDrive/Practicum/Models/y_hf_fail_target/xgb_y_hf_fail_target_metadata.json


## 5. Load both saved models into a shared inference registry

In [ ]:

LOADED_MODELS = {}

for target_col, cfg in MODEL_REGISTRY.items():
    pipeline = joblib.load(cfg["model_path"])
    with open(cfg["meta_path"], "r", encoding="utf-8") as f:
        meta = json.load(f)

    LOADED_MODELS[target_col] = {
        "pipeline": pipeline,
        "meta": meta,
        "features": meta["all_features"],
    }

print("Loaded targets:", list(LOADED_MODELS.keys()))
for target_col, obj in LOADED_MODELS.items():
    print(target_col, "-> feature count:", len(obj["features"]))


Loaded targets: ['y_fail_target', 'y_hf_fail_target']
y_fail_target -> feature count: 18
y_hf_fail_target -> feature count: 18


## 6. Shared inference helpers

In [ ]:

def predict_single_pair_for_target(
    input_record: dict,
    target_name: str,
    loaded_models: dict
) -> dict:
    """Score one target/conditioning pair for one model target."""

    pipeline = loaded_models[target_name]["pipeline"]
    expected_features = loaded_models[target_name]["features"]

    missing = [c for c in expected_features if c not in input_record]
    extra = [c for c in input_record if c not in expected_features]

    if missing:
        raise ValueError(f"Missing required features for {target_name}: {missing}")
    if extra:
        print(f"Warning: extra fields will be ignored for {target_name}: {extra}")

    X_new = pd.DataFrame([{c: input_record[c] for c in expected_features}])

    prob = float(pipeline.predict_proba(X_new)[0, 1])
    pred = int(prob >= 0.5)
    tier = assign_risk_tier(prob)

    return {
        "target_name": target_name,
        "prediction_binary": pred,
        "probability": prob,
        "risk_tier": tier
    }


def predict_single_pair_all_targets(
    input_record: dict,
    loaded_models: dict
) -> pd.DataFrame:
    """Score one pair against both saved model targets."""

    rows = []
    for target_name in loaded_models.keys():
        rows.append(predict_single_pair_for_target(input_record, target_name, loaded_models))
    return pd.DataFrame(rows).sort_values("probability", ascending=False).reset_index(drop=True)


def score_and_rank_targets_for_target(
    candidate_targets_df: pd.DataFrame,
    conditioning_barrier: dict,
    incident_context: dict,
    target_name: str,
    loaded_models: dict
) -> pd.DataFrame:
    """Rank many candidate target barriers for one selected model target."""

    pipeline = loaded_models[target_name]["pipeline"]
    expected_features = loaded_models[target_name]["features"]

    required_target_cols = [
        "control_id",
        "barrier_level_target",
        "lod_industry_standard_target",
        "pathway_sequence_target",
        "lod_numeric_target",
        "num_threats_in_lod_numeric_target",
    ]

    missing_target_cols = [c for c in required_target_cols if c not in candidate_targets_df.columns]
    if missing_target_cols:
        raise ValueError(f"candidate_targets_df is missing columns: {missing_target_cols}")

    records = []
    for _, row in candidate_targets_df.iterrows():
        rec = {
            "barrier_level_target": row["barrier_level_target"],
            "lod_industry_standard_target": row["lod_industry_standard_target"],
            "pathway_sequence_target": row["pathway_sequence_target"],
            "lod_numeric_target": row["lod_numeric_target"],
            "num_threats_in_lod_numeric_target": row["num_threats_in_lod_numeric_target"],

            "barrier_level_cond": conditioning_barrier["barrier_level_cond"],
            "lod_industry_standard_cond": conditioning_barrier["lod_industry_standard_cond"],
            "barrier_condition_cond": conditioning_barrier.get("barrier_condition_cond", "ineffective"),
            "pathway_sequence_cond": conditioning_barrier["pathway_sequence_cond"],
            "lod_numeric_cond": conditioning_barrier["lod_numeric_cond"],
            "num_threats_in_lod_numeric_cond": conditioning_barrier["num_threats_in_lod_numeric_cond"],

            "total_prev_barriers_incident": incident_context["total_prev_barriers_incident"],
            "total_mit_barriers_incident": incident_context["total_mit_barriers_incident"],
            "num_threats_in_sequence": incident_context["num_threats_in_sequence"],
            "flag_environmental_threat": incident_context["flag_environmental_threat"],
            "flag_electrical_failure": incident_context["flag_electrical_failure"],
            "flag_procedural_error": incident_context["flag_procedural_error"],
            "flag_mechanical_failure": incident_context["flag_mechanical_failure"],
        }
        records.append(rec)

    X_new = pd.DataFrame(records)[expected_features].copy()
    probs = pipeline.predict_proba(X_new)[:, 1]

    out = candidate_targets_df.copy()
    out[f"probability_{target_name}"] = probs
    out[f"risk_tier_{target_name}"] = pd.Series(probs).map(assign_risk_tier)
    out[f"risk_rank_{target_name}"] = out[f"probability_{target_name}"].rank(
        ascending=False,
        method="min"
    ).astype(int)

    return out.sort_values([f"probability_{target_name}", "control_id"], ascending=[False, True]).reset_index(drop=True)


def score_and_rank_targets_all_models(
    candidate_targets_df: pd.DataFrame,
    conditioning_barrier: dict,
    incident_context: dict,
    loaded_models: dict
) -> pd.DataFrame:
    """Rank many candidate target barriers and append scores from both saved models."""

    base = candidate_targets_df.copy()

    for target_name in loaded_models.keys():
        scored = score_and_rank_targets_for_target(
            candidate_targets_df=base,
            conditioning_barrier=conditioning_barrier,
            incident_context=incident_context,
            target_name=target_name,
            loaded_models=loaded_models
        )

        add_cols = [
            "control_id",
            f"probability_{target_name}",
            f"risk_tier_{target_name}",
            f"risk_rank_{target_name}",
        ]
        base = base.merge(scored[add_cols], on="control_id", how="left")

    sort_col = "probability_y_fail_target" if "probability_y_fail_target" in base.columns else base.columns[-1]
    return base.sort_values(sort_col, ascending=False).reset_index(drop=True)


## 7. Example: single-pair scoring against both models

In [ ]:

example_pair = {
    "lod_industry_standard_target": "2nd",
    "barrier_level_target": "preventive",
    "pathway_sequence_target": 2,
    "lod_numeric_target": 2,
    "num_threats_in_lod_numeric_target": 1,

    "lod_industry_standard_cond": "1st",
    "barrier_level_cond": "preventive",
    "barrier_condition_cond": "ineffective",
    "pathway_sequence_cond": 1,
    "lod_numeric_cond": 1,
    "num_threats_in_lod_numeric_cond": 2,

    "total_prev_barriers_incident": 5,
    "total_mit_barriers_incident": 3,
    "num_threats_in_sequence": 2,
    "flag_environmental_threat": 0,
    "flag_electrical_failure": 0,
    "flag_procedural_error": 1,
    "flag_mechanical_failure": 1,
}

predict_single_pair_all_targets(example_pair, LOADED_MODELS)


,target_name,prediction_binary,probability,risk_tier
0,y_hf_fail_target,1,0.877403,HIGH
1,y_fail_target,0,0.289285,LOW


## 8. Example: rank candidate target barriers and score both models side by side

In [ ]:

candidate_targets = pd.DataFrame([
    {
        "control_id": "T1",
        "barrier_level_target": "preventive",
        "lod_industry_standard_target": "2nd",
        "pathway_sequence_target": 2,
        "lod_numeric_target": 2,
        "num_threats_in_lod_numeric_target": 1,
    },
    {
        "control_id": "T2",
        "barrier_level_target": "mitigative",
        "lod_industry_standard_target": "recovery",
        "pathway_sequence_target": 3,
        "lod_numeric_target": 4,
        "num_threats_in_lod_numeric_target": 2,
    },
    {
        "control_id": "T3",
        "barrier_level_target": "preventive",
        "lod_industry_standard_target": "3rd",
        "pathway_sequence_target": 2,
        "lod_numeric_target": 3,
        "num_threats_in_lod_numeric_target": 1,
    },
])

conditioning_barrier = {
    "barrier_level_cond": "preventive",
    "lod_industry_standard_cond": "1st",
    "barrier_condition_cond": "ineffective",
    "pathway_sequence_cond": 1,
    "lod_numeric_cond": 1,
    "num_threats_in_lod_numeric_cond": 2,
}

incident_context = {
    "total_prev_barriers_incident": 5,
    "total_mit_barriers_incident": 3,
    "num_threats_in_sequence": 2,
    "flag_environmental_threat": 0,
    "flag_electrical_failure": 0,
    "flag_procedural_error": 1,
    "flag_mechanical_failure": 1,
}

score_and_rank_targets_all_models(
    candidate_targets_df=candidate_targets,
    conditioning_barrier=conditioning_barrier,
    incident_context=incident_context,
    loaded_models=LOADED_MODELS
)


,control_id,barrier_level_target,lod_industry_standard_target,pathway_sequence_target,lod_numeric_target,num_threats_in_lod_numeric_target,probability_y_fail_target,risk_tier_y_fail_target,risk_rank_y_fail_target,probability_y_hf_fail_target,risk_tier_y_hf_fail_target,risk_rank_y_hf_fail_target
0,T1,preventive,2nd,2,2,1,0.289285,LOW,1,0.877403,HIGH,1
1,T3,preventive,3rd,2,3,1,0.096692,LOW,2,0.156975,LOW,2
2,T2,mitigative,recovery,3,4,2,0.083809,LOW,3,0.109163,LOW,3
